In [1]:
%load_ext autoreload
%autoreload 2
import os
os.chdir("..")

os.environ["TORCHINDUCTOR_CACHE_DIR"] = "/root/torchinductor_cache"
os.environ["TORCHINDUCTOR_FX_GRAPH_CACHE"] = "1"

import torch
import diffusers
import transformers

transformers.utils.logging.set_verbosity_warning()
transformers.utils.logging.disable_progress_bar()
diffusers.utils.logging.set_verbosity_warning()
diffusers.utils.logging.disable_progress_bar()

from kandinsky import get_video_pipeline
from IPython.display import Video

FlashAttention 2 is found
Sage Attention is found


Pro  SFT

In [2]:
# import torch
# from diffusers import Kandinsky5T2VPipeline

# model_id = "kandinskylab/Kandinsky-5.0-T2V-Pro-distilled-5s-Diffusers"
# with torch.inference_mode():
#     pipe = Kandinsky5T2VPipeline.from_pretrained(model_id, dtype=torch.bfloat16)

In [3]:
# pipe = pipe.to("cuda")
# pipe.transformer.set_attention_backend("flex")                            # <--- Set attention bakend to Flex
# # pipe.enable_model_cpu_offload()                                           # <--- Enable cpu offloading for single GPU inference
# # pipe.transformer.compile(mode="max-autotune-no-cudagraphs", dynamic=True) # <--- Compile with max-autotune-no-cudagraphs

In [4]:
# # Generate video
# prompt = "A cat and a dog baking a cake together in a kitchen."
# negative_prompt = "Static, 2D cartoon, cartoon, 2d animation, paintings, images, worst quality, low quality, ugly, deformed, walking backwards"

# output = pipe(
#     prompt=prompt,
#     negative_prompt=negative_prompt,
#     height=768,
#     width=1024,
#     num_frames=121,  # ~5 seconds at 24fps
#     num_inference_steps=16,
#     guidance_scale=1.0,
# ).frames[0]


In [ ]:
pipe = get_video_pipeline(
    device_map={"dit": "cuda:0", "vae": "cuda:0", "text_embedder": "cuda:0"},
    conf_path="./configs/k5_pro_t2v_5s_sft_sd.yaml",
    mode='t2v',
    # attention_engine="nabla"
)

In [ ]:
# import torch
# from itertools import chain
# # switch memory layout to Torch's preferred, channels_last
# pipe.dit.to(memory_format=torch.channels_last)
# # set torch compile flags
# # config = torch._inductor.config
# # config.disable_progress = False  # show progress bar
# # config.conv_1x1_as_mm = True  # treat 1x1 convolutions as matrix muls
# # # adjust autotuning algorithm
# # config.coordinate_descent_tuning = True
# # config.coordinate_descent_check_all_directions = True
# # config.epilogue_fusion = False  # do not fuse pointwise ops into matmuls
# tr_blocks = chain(pipe.dit.text_transformer_blocks, pipe.dit.visual_transformer_blocks)
# for block in tr_blocks:
#     block.compile(mode="reduce-overhead", fullgraph=True, dynamic=True)
#     # block.compile(mode="max-autotune-no-cudagraphs", dynamic=True)

In [ ]:
# pipe.dit.text_transformer_blocks[0].compile(mode="reduce-overhead", fullgraph=True, dynamic=True)

In [ ]:
# torch.save(pipe.dit.example_input, "example_input.pt")
# example_input = torch.load("example_input.pt")

In [ ]:
# # text_embed, time_embed, text_rope, attention_mask = pipe.dit.example_input
# with torch.inference_mode():
#     res = pipe.dit.text_transformer_blocks[0](*example_input)

In [ ]:
with torch.inference_mode():
    out = pipe("A bear in a green hat.", time_length=5, width=768, height=512, save_path='./results/test07.mp4', num_steps=16, guidance_weight=1.0)
# Video('./test07.mp4')

Timesteps: tensor([1.0000, 0.9934, 0.9859, 0.9774, 0.9677, 0.9565, 0.9434, 0.9278, 0.9091,
        0.8861, 0.8571, 0.8197, 0.7692, 0.6977, 0.5882, 0.4000, 0.0000],
       device='cuda:0')


  0%|          | 0/16 [00:00<?, ?it/s]/root/kandinsky-5/.venv/lib/python3.13/site-packages/torch/_inductor/select_algorithm.py:3585: UserWarning: TypedStorage is deprecated. It will be removed in the future and UntypedStorage will be the only storage class. This should only matter to you if you are using storages directly.  To access UntypedStorage directly, use tensor.untyped_storage() instead of tensor.storage()
  current_out_size = out_base.storage().size()
Autotune Choices Stats:
{"num_choices": 7, "num_triton_choices": 7, "best_kernel": "triton_flex_attention_15", "best_kernel_desc": "BLOCKS_ARE_CONTIGUOUS=False, BLOCK_M=64, BLOCK_N=64, FLOAT32_PRECISION=\"'tf32'\", GQA_SHARED_HEADS=1, HAS_FULL_BLOCKS=True, IS_DIVISIBLE=False, OUTPUT_LOGSUMEXP=False, OUTPUT_MAX=False, PRESCALE_QK=False, QK_HEAD_DIM=128, QK_HEAD_DIM_ROUNDED=128, ROWS_GUARANTEED_SAFE=False, SAFE_HEAD_DIM=True, SM_SCALE=0.08838834764831843, SPARSE_KV_BLOCK_SIZE=64, SPARSE_Q_BLOCK_SIZE=64, USE_TMA=False, V_HEAD_DIM=12

KeyboardInterrupt: 

In [ ]:
# pipe = get_video_pipeline(
#     device_map={"dit": "cuda:0", "vae": "cuda:0", "text_embedder": "cuda:0"},
#     conf_path="./configs/k5_pro_t2v_10s_sft_sd.yaml", offload=True,
#     mode='t2v',
# )
# out = pipe("A bear in a blue hat.", time_length=10, width=768, height=512, save_path='./test07.mp4')
# Video('./test07.mp4')

In [ ]:
# pipe = get_video_pipeline(
#     device_map={"dit": "cuda:0", "vae": "cuda:0", "text_embedder": "cuda:0"},
#     conf_path="./configs/k5_pro_t2v_5s_sft_hd.yaml", offload=True,
#     mode='t2v',
# )
# out = pipe("A bear in a yellow hat.", time_length=5, width=1280, height=768, save_path='./test08.mp4')
# Video('./test08.mp4')

In [ ]:
# pipe = get_video_pipeline(
#     device_map={"dit": "cuda:0", "vae": "cuda:0", "text_embedder": "cuda:0"},
#     conf_path="./configs/k5_pro_t2v_10s_sft_hd.yaml", offload=True,
#     mode='t2v',
# )
# out = pipe("A bear in a red hat.", time_length=10, width=1280, height=768, save_path='./test09.mp4')
# Video('./test09.mp4')